# EONET API → PostgreSQL
## Notebook z Projektem bazy danych oraz importem bazy danych


<div style="text-align: center;">
    <img src="Onet.svg" width="900"  alt="Diagram bazy danych" />
</div>

### Encje (Tabele)
W schemacie występują następujące tabele:

* `event` - przechowuje podstawowe informacje o zdarzeniach (np. zjawiskach pobranych z API, z obciętym ID).
* `event_location` - przechowuje dane adresowe/lokalizacyjne (kraj, miasto) konkretnego zdarzenia.
* `geometry_type` - tabela słownikowa przechowująca typy obiektów geometrycznych.
* `geometry` - przechowuje dokładne współrzędne geograficzne, daty pomiaru oraz dodatkowe parametry (magnitude, unit).
* `category` - tabela słownikowa przechowująca kategorie zdarzeń.
* `source` - tabela słownikowa przechowująca źródła danych dla zdarzeń.
* `event_category` - tabela asocjacyjna łącząca zdarzenia z odpowiednimi kategoriami.
* `event_source` - tabela asocjacyjna łącząca zdarzenia z odpowiednimi źródłami.
* `event_geometry` - tabela asocjacyjna łącząca zdarzenia z przypisanymi im obiektami geometrycznymi.
* `import_log` - rejestr zdarzeń przechowujący informacje o procesach importu danych z API (logi).



### Lista  zależności:

| Zależność | Znaczenie |
| :--- | :--- |
| `event_location.event_id` &rarr; `event.id` | lokalizacja jest ściśle powiązana z konkretnym zdarzeniem |
| `geometry.type_id` &rarr; `geometry_type.id` | każdy rekord geometrii musi mieć określony typ (np. punkt, obszar) |
| `event` &rarr; `category` | zdarzenie jest przypisane do jedenego  lub wielu  kategorii |
| `event` &rarr; `source` | informacja o zdarzeniu pochodzi z jednego  lub  wielu źródeł |
| `event` &rarr; `geometry` | zdarzenie ma przypisane jedne  lub wiele dany współrzędnych |

## Skrypt tworzący baze danych

```sql

DROP TABLE IF EXISTS import_log;
DROP TABLE IF EXISTS event_category;
DROP TABLE IF EXISTS event_source;
DROP TABLE IF EXISTS event_geometry;
DROP TABLE IF EXISTS source;
DROP TABLE IF EXISTS category;
DROP TABLE IF EXISTS geometry;
DROP TABLE IF EXISTS geometry_type;
DROP TABLE IF EXISTS event_location;
DROP TABLE IF EXISTS event;

CREATE TABLE event (
	id INTEGER PRIMARY KEY, -- Przychodzi z API jako EONET_x zapisujemy tylko x
	title  VARCHAR(255) NOT NULL, -- Do przecinka z tego co przydchodzi w polu title
	description TEXT,
	start_date TIMESTAMP NOT NULL,
	end_date TIMESTAMP
);

CREATE TABLE event_location (
    id SERIAL PRIMARY KEY,
    event_id INTEGER NOT NULL, 
    country_code CHAR(2),
    country VARCHAR(100),
    state VARCHAR(100),
    city VARCHAR(100),

   
    CONSTRAINT fk_event_location_event
        FOREIGN KEY (event_id) 
        REFERENCES event(id)
		ON UPDATE CASCADE 
        ON DELETE CASCADE
);

CREATE TABLE geometry_type (
	id SERIAL PRIMARY KEY,
	type VARCHAR(100) NOT NULL
);

CREATE TABLE geometry (
	id SERIAL PRIMARY KEY, -- Własny z sql
	latitude FLOAT NOT NULL CHECK (90 >= latitude AND latitude >= -90), --latitude i longitude są w takich zakresach
	longitude FLOAT NOT NULL CHECK (180 >= longitude AND longitude >= -180),
	type_id INTEGER NOT NULL,
	magnitude FLOAT,
	unit FLOAT,
	measure_date TIMESTAMP NOT NULL,

	CONSTRAINT fk_geometry_type_geometry
		FOREIGN KEY (type_id)
		REFERENCES geometry_type(id)
		ON UPDATE CASCADE
		ON DELETE RESTRICT --nie chcemy usunąć typu punktu który jest zapisany w bazie
);

CREATE TABLE category (
	id SERIAL PRIMARY KEY,
	abbreviation VARCHAR(50) NOT NULL,
	title VARCHAR(255) NOT NULL,
	description TEXT
);

CREATE TABLE source (
	id SERIAL PRIMARY KEY,
	abbreviation VARCHAR(50) NOT NULL,
	title VARCHAR(255) NOT NULL,
	description TEXT
	
);

CREATE TABLE event_category (
	event_id INTEGER NOT NULL,
	category_id INTEGER NOT NULL,

	CONSTRAINT pk_event_category
		PRIMARY KEY (event_id, category_id),
		
	CONSTRAINT fk_event_category_event
		FOREIGN KEY (event_id)
		REFERENCES event(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE,

	CONSTRAINT fk_event_category_category
		FOREIGN KEY (category_id)
		REFERENCES category(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE
);

CREATE TABLE event_source (
	event_id INTEGER  NOT NULL,
	source_id INTEGER NOT  NULL,

	
	CONSTRAINT pk_event_source
		PRIMARY KEY (event_id, source_id),

	CONSTRAINT fk_event_source_event
		FOREIGN KEY (event_id)
		REFERENCES event(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE,

	CONSTRAINT fk_event_source_source
		FOREIGN KEY (source_id)
		REFERENCES source(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE
);

CREATE TABLE event_geometry (
	event_id INTEGER NOT NULL,
	geometry_id INTEGER NOT NULL,


	CONSTRAINT pk_event_geometry
		PRIMARY KEY (event_id, geometry_id),

	CONSTRAINT fk_event_geometry_event
		FOREIGN KEY (event_id)
		REFERENCES event(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE,

	CONSTRAINT fk_event_geometry
		FOREIGN KEY (geometry_id)
		REFERENCES geometry(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE

);

CREATE TABLE import_log (
    id SERIAL PRIMARY KEY,
    imported_at TIMESTAMP NOT NULL,
    apiname VARCHAR(255) NOT NULL,
    endpoint VARCHAR(255) NOT NULL,
    records_received INTEGER NOT NULL,
    records_saved INTEGER NOT NULL,
    status VARCHAR(100) NOT NULL,
    error_message TEXT NOT NULL
);
```

## Import bibliotek

In [ ]:
from __future__ import annotations
from typing import Any
from time import sleep


from sqlalchemy import URL, create_engine, text
from sqlalchemy.exc import SQLAlchemyError
import pandas as pd
import reverse_geocode as rg
import requests

## Tworzenie schematu bazy danych

##### Struktura pliku konfiguracyjnego (conf.txt):
DB_USER=moj_uzytkownik\
DB_PASSWORD=moje_super_tajne_haslo\
DB_HOST=localhost\
DB_PORT=5432\
DB_NAME=moja_baza_danych

In [ ]:


config = {}
with open("conf.txt", "r", encoding="utf-8") as f:
    for linia in f:
        linia = linia.strip()
        if not linia or linia.startswith("#"):
            continue
        klucz, wartosc = linia.split("=", 1)
        config[klucz.strip()] = wartosc.strip()


db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=config.get("DB_USER"),
    password=config.get("DB_PASSWORD"),
    host=config.get("DB_HOST"),
    port=config.get("DB_PORT"),      
    database=config.get("DB_NAME"),
)


from scripts import schema



engine = create_engine(db_url)

try:
    conn = engine.connect()
    print("[INFO] POŁACZONO DO BAZY DANYCH")
except SQLAlchemyError as e:
    print(f"[ERROR] BŁĄD POŁĄCZENIA DO BAZY DANYCH {e}")

res = conn.execute(text(schema))
conn.commit()
print("[INFO] UTWORZONO BAZĘ DANYCH")
conn.close()

## Pobieranie danych z API


In [ ]:
from  fetch_api import fetch_events, fetch_categories, fetch_source
DAYS = 100
LIMIT = 100
events = fetch_events(DAYS, LIMIT)
print(events[0:3])
sources = fetch_source()
print(sources[0:3])
categories = fetch_categories()
print(categories[0:3])

## Zapisywanie danych do tabel

In [ ]:
from import_danych import import_event, import_categories, import_sources

import_sources(sources, engine)
import_categories(categories, engine)
import_event(events, engine)



## Weryfikacja zawartości bazy danych

## Skrypty do analizy  lub  raportów

####  Najpopularniejsze  zjawiska
```sql
SELECT 
    c.title AS kategoria, 
    c.abbreviation AS skrot,
    COUNT(ec.event_id) AS liczba_zdarzen
FROM category c
JOIN event_category ec ON c.id = ec.category_id
GROUP BY c.title, c.abbreviation
ORDER BY liczba_zdarzen DESC
LIMIT 10;
```

In [ ]:
from scripts import analyze_1 as most_popular

with engine.begin() as conn:
    res = pd.read_sql(most_popular, con=conn)
    display(res)

#### Liczba zjawisk w poszczególnych krajach
```sql
SELECT 
    el.country AS kraj, 
    COUNT(e.id) AS liczba_zdarzen
FROM event e
JOIN event_location el ON e.id = el.event_id
WHERE el.country IS NOT NULL
GROUP BY el.country
ORDER BY liczba_zdarzen DESC;
```

In [ ]:
from scripts import analyze_2 as by_country

with engine.begin() as conn:
    res = pd.read_sql(by_country, con=conn)
    display(res)

#### Aktywne zjawiska
```sql
SELECT 
    e.title AS nazwa_zdarzenia,
    e.start_date AS data_rozpoczecia,
    s.title AS zrodlo_danych
FROM event e
JOIN event_source es ON e.id = es.event_id
JOIN source s ON es.source_id = s.id
WHERE e.end_date IS NULL
ORDER BY e.start_date DESC;
```

In [ ]:
from scripts import analyze_3 as active

with engine.begin() as conn:
    res = pd.read_sql(active, con=conn)
    display(res)

## Analiza Danych i Wizualizacja